In [1]:
!pip install azure-ai-projects azure-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 892.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 6.0 MB/s eta 0:00:00


In [1]:
pip install --upgrade openai azure-ai-projects azure-identity

In [2]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("azure-key")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://kushagra22scse1011704-3044-resou.openai.azure.com/openai/v1"
)

In [3]:
# 1. Define the input data
medical_report_data = """
Patient Name: John Smith
Age: 45
Gender: Male

Test Results:
Hemoglobin: 11.2 g/dL
WBC Count: 14500 /uL
Blood Sugar (Fasting): 132 mg/dL
Cholesterol: 245 mg/dL
Vitamin D: 18 ng/mL

Doctor Notes:
Patient reports fatigue, mild fever, and weakness.
"""

# 2. Extract structured medical report data
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": """Extract the following details from the medical report and return ONLY valid JSON:

                        - Patient Name
                        - Age
                        - Gender
                        - Summary
                        - Abnormal Values
                        - Possible Conditions
                        - Recommendations
                        """
                    },
                    {
                        "type": "input_text",
                        "text": f"MEDICAL REPORT:\n{medical_report_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED MEDICAL REPORT DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED MEDICAL REPORT DATA ---
[ResponseOutputText(annotations=[], text='```json\n{\n  "Patient Name": "John Smith",\n  "Age": 45,\n  "Gender": "Male",\n  "Summary": "The patient presents with fatigue, mild fever, and weakness. Abnormal lab results include low hemoglobin, elevated WBC count, high fasting blood sugar, high cholesterol, and low vitamin D.",\n  "Abnormal Values": {\n    "Hemoglobin": "11.2 g/dL (Low)",\n    "WBC Count": "14500 /uL (High)",\n    "Blood Sugar (Fasting)": "132 mg/dL (High)",\n    "Cholesterol": "245 mg/dL (High)",\n    "Vitamin D": "18 ng/mL (Low)"\n  },\n  "Possible Conditions": [\n    "Anemia",\n    "Infection or Inflammatory process",\n    "Diabetes Mellitus",\n    "Hypercholesterolemia",\n    "Vitamin D deficiency"\n  ],\n  "Recommendations": [\n    "Further evaluation for anemia, including iron studies",\n    "Investigate source of infection or inflammation",\n    "Repeat fasting blood sugar and consider diabetes management",\n    "Lipid profile 

In [7]:
import json
import re

output_text_item = response.output[0].content[0]

raw_json_string = output_text_item.text if hasattr(output_text_item, "text") else output_text_item.get("text", "")

print("RAW:", repr(raw_json_string))

if not raw_json_string.strip():
    raise ValueError("Empty response from model")

# remove markdown formatting
cleaned = re.sub(r"^```json|^```|```$", "", raw_json_string.strip()).strip()

extracted_data = json.loads(cleaned)

print(json.dumps(extracted_data, indent=2))

RAW: '```json\n{\n  "Patient Name": "John Smith",\n  "Age": 45,\n  "Gender": "Male",\n  "Summary": "The patient presents with fatigue, mild fever, and weakness. Abnormal lab results include low hemoglobin, elevated WBC count, high fasting blood sugar, high cholesterol, and low vitamin D.",\n  "Abnormal Values": {\n    "Hemoglobin": "11.2 g/dL (Low)",\n    "WBC Count": "14500 /uL (High)",\n    "Blood Sugar (Fasting)": "132 mg/dL (High)",\n    "Cholesterol": "245 mg/dL (High)",\n    "Vitamin D": "18 ng/mL (Low)"\n  },\n  "Possible Conditions": [\n    "Anemia",\n    "Infection or Inflammatory process",\n    "Diabetes Mellitus",\n    "Hypercholesterolemia",\n    "Vitamin D deficiency"\n  ],\n  "Recommendations": [\n    "Further evaluation for anemia, including iron studies",\n    "Investigate source of infection or inflammation",\n    "Repeat fasting blood sugar and consider diabetes management",\n    "Lipid profile monitoring and dietary modifications for cholesterol",\n    "Vitamin D sup

In [8]:
from azure.storage.blob import BlobServiceClient

# ✅ Correct connection string
AZURE_STORAGE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=kushagerastorage101;AccountKey=KkuAKXWPKjkMLvcd9X4P0isEIMVBci3AaamZjH2nuv2Glw/FVEi58JaC2artNGnsMw8+6rcH0u/0+AStZ9Jtzg==;EndpointSuffix=core.windows.net"

blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "container101"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [9]:
import datetime

file_name = f"medicalreport_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [10]:
blob_client = blob_service_client.get_blob_client(
    container="container101",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F88150FC25D"',
 'last_modified': datetime.datetime(2026, 4, 21, 9, 26, 35, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\xdb\xf0U\xcd\x07\xed\xa2\xfa\xde.\xdb\xb2l\x9d}?'),
 'client_request_id': '30d5f70e-3d64-11f1-a720-0242ac1c000c',
 'request_id': 'bfc2bafd-501e-0050-7f70-d14937000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 9, 26, 34, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [11]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: medicalreport_20260421_092553.txt
